# Pipeline D: Campaign ROI Attribution

**Object Name:** Social Media Revenue Lift & ROI Explanatory Model

**Goal:** Holding all else equal, what is the marginal donation revenue generated by different campaign types and post characteristics?

**Business Problem:** HealingWings spends money on boosting social media posts and time on content creation. Most nonprofits cannot link a specific donation to a specific post. Because we have `referral_post_id`, we can quantify which campaign strategies actually "move money" vs. just generating vanity engagement metrics like likes or shares.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Setup Data Paths
REPO_ROOT = Path.cwd().parent.parent
DATA_DIR = REPO_ROOT / "data" / "lighthouse_csv_v7"

def load_table(name):
    fp = DATA_DIR / f"{name}.csv"
    return pd.read_csv(fp) if fp.exists() else pd.DataFrame()

donations = load_table("donations")
posts = load_table("social_media_posts")

## Data Wrangling: Attribution Join

We roll up donation revenue to the individual post level. This creates a dataset where each row is a `post_id` and the target is `total_revenue_generated`.

In [ ]:
donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')

# Attribution via referral_post_id
attributed_revenue = donations.groupby('referral_post_id')['amount'].sum().reset_index()
attributed_revenue.columns = ['post_id', 'revenue']

# Join with post characteristics
df = posts.merge(attributed_revenue, on='post_id', how='left').fillna({'revenue': 0})

print("Attribution Join Complete. Shape:", df.shape)
print("Top Revenue Generating Posts:")
display(df.sort_values('revenue', ascending=False).head(5))

## Explanatory Model: Ridge Regression

We model **Revenue** as a function of post attributes. This tells us the "marginal lift" of factors like platform, boosting, and sentiment.

In [ ]:
num_feats = ['engagement_rate', 'reach', 'boost_budget_php']
cat_feats = ['platform', 'post_type', 'sentiment_tone', 'campaign_name']

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0)), ('scaler', StandardScaler())]), num_feats),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='unknown')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_feats)
])

ridge_pipe = Pipeline([('prep', preprocessor), ('reg', Ridge(alpha=1.0))])

X = df[num_feats + cat_feats]
y = df['revenue']

ridge_pipe.fit(X, y)

fnames = list(ridge_pipe.named_steps['prep'].get_feature_names_out())
coefs = ridge_pipe.named_steps['reg'].coef_

summary = pd.Series(coefs, index=fnames).sort_values(ascending=False)
print("Revenue Lift Factors (Ridge Coefficients):")
print(summary.head(10))

## Causal Discussion & ROI Analysis

**Marginal ROI of Boosting:** The coefficient for `boost_budget_php` represents the marginal revenue generated for every 1 unit of currency spent on boosting, holding content and platform constant.

**Endogeneity & Selection Bias:** One causal concern is that "better" posts (higher quality content) get more budget and more donations. By including content characteristics (`sentiment_tone`, `post_type`) and `engagement_rate` as covariates, we attempt to isolate the pure "lift" of the boosting spend vs. the content quality.

**Platform ROI:** Which platform (Facebook vs. Instagram) is more efficient at converting engagement to cash? The platform coefficients provide the answer.